In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
from scipy.special import expit

import pymc as pm
import pymc.sampling.jax as pmjax
import preliz as pz
import pytensor.tensor as pt
import arviz as az
import arviz_plots as azp
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from sklearn.calibration import calibration_curve

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
print("pymc: "+pm.__version__)
print("pz: "+pz.__version__)

pandas: 3.0.1
numpy: 2.4.3
pymc: 6.2.0
pz: 0.27.1


In [20]:
df = pd.read_csv('data/shot_probs_data.csv')
df.head()

,distance,angle,shot_type_numeric,shot_type,is_behind_backboard,set_name,is_made
0,0.351418,0.220458,0,JUMPER,0,train,0
1,0.015228,0.358510,1,LAYUP,0,train,1
2,0.078598,0.050241,0,JUMPER,0,train,0
3,0.097535,0.202042,0,JUMPER,0,train,0
4,0.078598,0.050241,0,JUMPER,0,train,1


In [21]:
def make_dataset(df:pd.DataFrame, set_name:str):
    dff = df.loc[df['set_name']==set_name].drop(['set_name','is_made','shot_type','shot_type_numeric'],axis=1).reset_index(drop=True)
    target = df.loc[df['set_name']==set_name]['is_made'].values
    shot_name_vals = df.loc[df['set_name']==set_name]['shot_type'].values
    return dff, target, shot_name_vals

X_train_mm, y_train, shot_train = make_dataset(df, 'train')
X_val_mm, y_val, shot_val = make_dataset(df, 'val')
X_query_mm, y_query, shot_query = make_dataset(df, 'query')
X_test_mm, y_test, shot_test = make_dataset(df, 'test')

print(X_train_mm.shape, X_val_mm.shape, X_query_mm.shape, X_test_mm.shape)

(107387, 3) (46024, 3) (32874, 3) (32874, 3)


In [22]:
def sample(
    model: pm.Model, draws: int = 1000, tune: int = 1000, chains: int = 4, target_accept: float = 0.95, random_seed: int = RANDOM_SEED
):
    """
    Fit model using MCMC.

    Parameters
    ----------
    model: pm.Model
        PyMC model object.
    draws : int
        Number of draws to keep from the sampling process.
    tune : int
        Number of tuning steps to take before sampling.
    chains : int
        Number of chains to sample.
    target_accept : float
        Target acceptance probability for step size adaptation.
    random_seed : int
        Seed for randomness.
    """
    with model:
        trace = pmjax.sample_numpyro_nuts(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            idata_kwargs={"log_likelihood": False}
        )
    return trace

def compute_log_likelihood(model: pm.Model, trace: az.InferenceData) -> None:
    """Wrapper to compute elemwise log_likelihood of model given InferenceData with posterior group
    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling
    """
    with model:
        pm.compute_log_likelihood(trace)
    return None

def sample_posterior_pred(model: pm.Model, trace: az.InferenceData) -> az.InferenceData:
    """Generates samples from the posterior predictive distribution for model checks

    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling

    Returns:
        az.InferenceData: An ArviZ InferenceData object containing the posterior predictive samples.
    """
    with model:
        spp = pm.sample_posterior_predictive(
            trace,
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )
    return spp

/opt/homebrew/Caskroom/miniforge/base/envs/pie/lib/python3.12/site-packages/arviz/__init__.py:110: MigrationWarning: arviz.InferenceData is no longer available on the arviz package; ArviZ now uses xarray's DataTree for the same role. See the migration guide: https://python.arviz.org/en/latest/user_guide/migration_guide.html#datatree
  warnings.warn(


In [23]:
m_dist, c_dist = pm.gp.hsgp_approx.approx_hsgp_hyperparams(
    x_range=[0, 1],
    lengthscale_range=[0.1, 0.3],
    cov_func="matern52",
)
m_angle, c_angle = pm.gp.hsgp_approx.approx_hsgp_hyperparams(
    x_range=[0, 1],
    lengthscale_range=[0.1, 0.3],
    cov_func="matern52",
)
m = [m_dist, m_angle]
c = max(c_dist, c_angle)
print(m)
print(c)

[32, 32]
2.4599999999999995


In [24]:
geo_vals = np.ascontiguousarray(X_train_mm[['distance', 'angle']].values.astype(np.float32))
behind_vals = np.ascontiguousarray(X_train_mm['is_behind_backboard'].values.astype(np.float32))
geo_var_names = ['distance','angle']
coords = {
        "geo_variables":np.array(geo_var_names),
        "obs_id": np.arange(len(y_train))
    }
with pm.Model(coords=coords) as model:
    shot_made = pm.Data("shot_made", y_train, dims=("obs_id",))
    geometry_data = pm.Data("geometry_data", geo_vals, dims=("obs_id","geo_variables"))
    behind_data = pm.Data("behind_data", behind_vals, dims=("obs_id",))

    eta = pm.Exponential("eta", 1)
    ell = pz.maxent(distribution=pz.InverseGamma(), lower=0.1, upper=0.3, mass=0.95, plot=False).to_pymc("ell", shape=2)
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=ell)
    gp = pm.gp.HSGP(
            m=m,
            c=c,
            mean_func=pm.gp.mean.Constant(-3),
            cov_func=cov_func,
        )
    f = gp.prior("f", X=geometry_data)
    beta_behind = pm.Normal("beta_behind", mu=0, sigma=1)
    logit_p = pm.Deterministic("logit", f + beta_behind * behind_data, dims=("obs_id",))
    pm.Bernoulli("shot_lkhood", logit_p=logit_p, observed=shot_made, dims=("obs_id",))


In [25]:
trace = sample(model)

KeyboardInterrupt: 

In [26]:
with model:
    approx = pm.fit(
        method="advi", # fullrank_advi
        n=30000,
        obj_optimizer=pm.adam(learning_rate=0.01),
        random_seed=RANDOM_SEED,
    )
    trace = approx.sample(1000)

Output()

SystemError: CPUDispatcher(<function numba_funcified_fgraph at 0x375a2fd80>) returned a result with an exception set